# Shadow-Net · SOREL-20M — Validación completa (acceso + correspondencia fila↔sha)
Master — notebook único e importable. Dos fases con puertas duras:
**Fase 0**: H0–H5 (entorno, acceso, Range, ZIP64, 1 vector real 2381, cruce meta.db).
**Fase 1**: H6–H7 (orden fila↔sha + etiqueta) por prueba estructural + semántica.
Contrato: `X = [EMBER_2381 | OVERLAY_N]` — las 2381 nunca se tocan. N sin fijar (H8).
Si cualquier puerta falla → abortar, no pasar a 7M/scaler/FFNN.


In [ ]:
# CELDA 0 — Constantes unificadas + flag de reanudación.
import os
SKIP_VALIDATED = True  # si los artefactos ya existen en disco, se omiten descargas y reconstrucción
TRAIN_SPLIT = 1543542570.0  # config.py oficial sophos/SOREL-20M
VAL_SPLIT = 1547279640.0
EXPECTED_NPZ_ROWS = 12699013
EXPECTED_NPZ_COLS = 2381
NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"
LMDB = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/ember_features/data.mdb"
META_URL = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/meta.db"
META_SIZE = 3788979200
MISSING_URL = "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json"
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
print("WORK =", WORK, "| SKIP_VALIDATED =", SKIP_VALIDATED)


In [ ]:
# CELDA 0 — Detección de entorno (sin asumir P100 / 12h / cuota). Solo observa y decide.
import os, shutil, subprocess, sys

def sh(cmd):
    try: return subprocess.check_output(cmd, shell=True, text=True, timeout=15).strip()
    except Exception as e: return f"NA ({e})"

print("python:", sys.version.split()[0])
print("nvidia-smi:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>&1 | head -5"))
print("RAM:", sh("free -g | head -3"))
print("df /kaggle/working:", sh("df -h /kaggle/working 2>&1 | tail -2"))
print("df /tmp:", sh("df -h /tmp 2>&1 | tail -2"))
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), end="")
    if torch.cuda.is_available():
        print(f" | {torch.cuda.get_device_name(0)} | VRAM={torch.cuda.get_device_properties(0).total_memory/2**30:.1f}GB")
    else: print(" | solo CPU — el pipeline debe seguir siendo válido (más lento)")
except ImportError:
    print("torch: no instalado — instalar en fase de entrenamiento, no aquí")

# Puerta de espacio: meta.db pesa ~3.79GB. Solo avisa, no descarga todavía.
free_b = shutil.disk_usage("/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp").free
print(f"espacio libre destino: {free_b/2**30:.1f} GB (meta.db necesita ~3.8GB + margen)")
assert free_b > 6*2**30, "FAIL: sin espacio mínimo para meta.db + trabajo. Abortar."
print("PASS celda 0: entorno caracterizado, sin supuestos de GPU/horas/cuota.")

In [ ]:
# CELDA 1 — Acceso HTTPS/S3 al bucket público (solo HEAD, cero descargas pesadas).
# Hipótesis H1: el bucket sorel-20m es público en us-west-2 y responde sin credenciales.
import urllib.request

BASE = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020"
TARGETS = {
    "meta.db": f"{BASE}/processed-data/meta.db",                       # ~3.79GB sqlite
    "ember_data.mdb": f"{BASE}/processed-data/ember_features/data.mdb", # ~72GB LMDB (NO descargar)
    "train.npz": f"{BASE}/lightGBM-features/train-features.npz",        # ~121GB (NO descargar)
    "missing": "https://github.com/sophos/SOREL-20M/raw/master/shas_missing_ember_features.json",
}

def head(url, timeout=30):
    req = urllib.request.Request(url, method="HEAD")
    r = urllib.request.urlopen(req, timeout=timeout)
    return r.status, dict(r.headers)

for name, url in TARGETS.items():
    try:
        st, h = head(url)
        print(f"{name}: HTTP {st} | len={h.get('Content-Length')} | range={h.get('Accept-Ranges')} | etag={h.get('ETag')}")
        assert st == 200 and h.get("Accept-Ranges") == "bytes", f"FAIL H1 en {name}"
    except Exception as e:
        print(f"{name}: FAIL ({e})")
        raise AssertionError(f"H1 no verificada para {name}: {e}")
print("PASS celda 1 (H1): bucket público accesible, Accept-Ranges: bytes en los tres artefactos.")

In [ ]:
# CELDA 2 — Range Requests reales (hipótesis H2: el servidor honra Range con 206).
# Sin 206 no hay streaming posible y el diseño chunked queda invalidado.
import urllib.request

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=30):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    r = urllib.request.urlopen(req, timeout=timeout)
    return r.status, r.headers.get("Content-Range"), r.read()

st, cr, magic = get_range(NPZ, 0, 3)
print("head-range:", st, cr, magic)
assert st == 206, "FAIL H2: el servidor no devolvió 206"
assert magic == b"PK\x03\x04", f"FAIL H2: magia ZIP ausente: {magic!r}"
TOTAL = int(cr.split("/")[1])
print(f"tamaño total declarado: {TOTAL/2**30:.1f} GiB")
st2, cr2, tail_probe = get_range(NPZ, TOTAL-64, TOTAL-1)
assert st2 == 206 and len(tail_probe) == 64, "FAIL H2: Range de cola no honrado"
print("PASS celda 2 (H2): Range HEAD y TAIL honrados con 206. Streaming viable.")

In [ ]:
# CELDA 3 — Inspección ZIP/NPZ por cola (hipótesis H3: el .npz es un ZIP cuyo
# Central Directory + EOCD permiten listar entradas con Range GET, sin descargar 121GB).
# OJO: fichero >4GB => ZIP64. El EOCD clásico trae cd_off=0xFFFFFFFF y el offset real
# vive en el ZIP64 EOCD (localizado por PK\x06\x07). Sin este paso, el CD se lee de un
# offset basura y la firma PK\x01\x02 falla (ese era el bug).
import io, struct, urllib.request

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=60):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    r = urllib.request.urlopen(req, timeout=timeout)
    assert r.status == 206, f"Range no honrado: {r.status}"
    data = r.read()
    assert len(data) == b - a + 1, f"truncado: {len(data)} vs {b-a+1}"
    return data

TOTAL = 121046992510  # re-verificado en celda 2 vía Content-Range; se re-deriva abajo si cambia
tail = get_range(NPZ, TOTAL-131072, TOTAL-1)  # últimos 128KB: locator + EOCD + CD final
eocd = tail.rfind(b"PK\x05\x06")
assert eocd != -1, "FAIL H3: EOCD no encontrado en los últimos 128KB"
(n_disk, n_cd, n_entries, n_total, cd_size, cd_off, _) = struct.unpack("<HHHHIIH", tail[eocd+4:eocd+22])
print(f"EOCD clásico: n_total={n_total} cd_size={cd_size:#x} cd_off={cd_off:#x}")
if cd_off == 0xFFFFFFFF:  # ZIP64: resolver offset real de 64 bits
    loc = tail.rfind(b"PK\x06\x07")
    assert loc != -1, "FAIL H3: EOCD clásico con placeholder ZIP64 pero sin locator"
    (_, zoff, _) = struct.unpack("<IQI", tail[loc+4:loc+20])
    z = get_range(NPZ, zoff, zoff+55)
    assert z[:4] == b"PK\x06\x06", "FAIL H3: firma ZIP64 EOCD ausente"
    (_, _, _, _, _, _, ntot64, cd_size, cd_off) = struct.unpack("<QHHIIQQQQ", z[4:56])
    n_total = ntot64
    print(f"ZIP64 EOCD: n_total={n_total} cd_size={cd_size} cd_off={cd_off}")
assert n_total >= 1 and cd_size > 0, "FAIL H3: inventario vacío"
cd_blob = get_range(NPZ, cd_off, cd_off+cd_size-1)
# Central Directory: sig(4) madeby(2) | needed(2) flags(2) comp(2) time(2) date(2) crc(4)
# csize(4) usize(4) nlen(2) elen(2) clen(2) disk(2) iattr(2) eattr(4) lho(4) => "<HHHHHIIIHHHHHII" desde pos+6
entries, pos = [], 0
while pos < len(cd_blob):
    assert cd_blob[pos:pos+4] == b"PK\x01\x02", f"FAIL H3: firma CD corrupta en offset {pos}"
    (vneed, flags, comp, mt, md, crc, csz32, usz32, nlen, elen, clen, disk, iattr, eattr, lho32) = struct.unpack("<HHHHHIIIHHHHHII", cd_blob[pos+6:pos+46])
    name = cd_blob[pos+46:pos+46+nlen].decode()
    extra = cd_blob[pos+46+nlen:pos+46+nlen+elen]
    csize, usize, lho = csz32, usz32, lho32
    j = 0  # extra ZIP64 (id 0x0001) rellena los campos que sean 0xFFFFFFFF, en orden usize/csize/lho
    while j + 4 <= len(extra):
        hid, dsz = struct.unpack("<HH", extra[j:j+4])
        if hid == 0x0001:
            nq = dsz // 8
            vals = struct.unpack("<" + "Q"*nq, extra[j+4:j+4+8*nq])
            vi = 0
            if usize == 0xFFFFFFFF: usize = vals[vi]; vi += 1
            if csize == 0xFFFFFFFF: csize = vals[vi]; vi += 1
            if lho == 0xFFFFFFFF: lho = vals[vi]; vi += 1
            break
        j += 4 + dsz
    entries.append({"name": name, "compress": comp, "csize": csize, "usize": usize, "lho": lho})
    pos += 46 + nlen + elen + clen
arr = [e for e in entries if e["name"].endswith(".npy")]
for e in entries: print(e)
assert arr, "FAIL H3: el npz no contiene ningún .npy"
STORED = all(e["compress"] == 0 for e in arr)
print(f"compresión: {'STORED (row-Range posible)' if STORED else 'DEFLATED (row-Range IMPOSIBLE: ver fallback)'}")
print("PASS celda 3 (H3): inventario ZIP obtenido por Range. Entradas:", [e["name"] for e in entries])


In [ ]:
# CELDA 4 — Un vector EMBER real por streaming (hipótesis H4: row-slice por Range es posible
# SOLO si el método es STORED; si es DEFLATED esta celda debe FALLAR explícitamente y activar
# el plan B: shards Parquet pre-materializados fuera de Kaggle. No fingir éxito.
# También valida len==2381 y ausencia de NaN/Inf (H5)."
import struct, urllib.request, numpy as np

NPZ = "http://sorel-20m.s3.amazonaws.com/09-DEC-2020/lightGBM-features/train-features.npz"

def get_range(url, a, b, timeout=60):
    req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"})
    r = urllib.request.urlopen(req, timeout=timeout)
    assert r.status == 206, f"Range no honrado: {r.status}"
    return r.read()

# 4a. Local File Header de la primera entrada .npy:
# sig(4) ver(2) flags(2) comp(2) time(2) date(2) crc(4) csize(4) usize(4) nlen(2) elen(2)
lho = arr[0]["lho"]  # reutiliza arr de la celda 3
lh = get_range(NPZ, lho, lho+29)
assert lh[:4] == b"PK\x03\x04", "FAIL H4: firma Local File Header ausente"
comp = struct.unpack("<H", lh[8:10])[0]
nlen = struct.unpack("<H", lh[26:28])[0]
elen = struct.unpack("<H", lh[28:30])[0]
data_off = lho + 30 + nlen + elen
print(f"método={comp} (0=STORED,8=DEFLATED) | data_off={data_off} | csize={arr[0]['csize']}")
if comp != 0:
    raise AssertionError("H4 NO verificada: .npy con DEFLATE — row-Range imposible. Activar plan B (shards Parquet). ABORTAR aquí.")

# 4b. Cabecera .npy (magia 6B + ver 2B + hlen 2B) para obtener shape/dtype/fortran.
npy_head = get_range(NPZ, data_off, data_off+127)
assert npy_head[:6] == b"\x93NUMPY", "FAIL H4: magia NPY ausente"
hlen = struct.unpack("<H", npy_head[8:10])[0]
hdr = get_range(NPZ, data_off, data_off+10+hlen-1)[10:].decode("latin1")
print("npy header:", hdr.strip())
d = eval(hdr)  # formato controlado por numpy; validado arriba por magia
shape, dtype, order = tuple(d["shape"]), np.dtype(d["descr"]), d["fortran_order"]
assert order is False, "FAIL H4: array Fortran — recalcular strides"
n_rows, n_cols = shape
print(f"shape={shape} dtype={dtype}")
assert n_cols == 2381, f"FAIL H4: columnas={n_cols}, se exigen 2381 exactas"
row_bytes = n_cols * dtype.itemsize
payload = data_off + 10 + hlen

# 4c. Fila 0 por Range puro (cero descargas masivas: solo row_bytes).
row0 = np.frombuffer(get_range(NPZ, payload, payload+row_bytes-1), dtype=dtype)
assert row0.shape == (2381,), f"FAIL H4: fila={row0.shape}"
assert np.all(np.isfinite(row0)), "FAIL H5: NaN/Inf en vector real"
print(f"PASS celdas 4 (H4+H5): fila0 real len=2381, finita. min={row0.min():.4g} max={row0.max():.4g} mean={row0.mean():.4g}")
print("Bloques canónicos: hist=row0[0:256] ent=row0[256:512] str=row0[512:616] gen=row0[616:626] ...")


In [ ]:
# FASE 0 — meta.db (descarga única 3.8GB, con reanudación) + esquema + conteo train.
# sqlite exige fichero local: no hay Range posible. H6/H7 quedan PENDIENTES hasta la Fase 1.
import sqlite3, urllib.request
have = os.path.getsize(META) if os.path.exists(META) else 0
print("meta.db presente:", have, "/", META_SIZE)
if have != META_SIZE:
    print("descargando meta.db ...")
    req = urllib.request.Request(META_URL)
    if have > 0:
        req.add_header("Range", "bytes=%d-" % have)
    r = urllib.request.urlopen(req, timeout=120)
    mode = "ab" if have > 0 and r.status == 206 else "wb"
    f = open(META, mode)
    done = os.path.getsize(META) if mode == "ab" else 0
    while True:
        b = r.read(8*2**20)
        if not b: break
        f.write(b); done += len(b)
    f.close()
    assert os.path.getsize(META) == META_SIZE, "FAIL: meta.db incompleto"
con = sqlite3.connect(META)
cols = [rr[1] for rr in con.execute("PRAGMA table_info(meta)").fetchall()]
print("columnas meta:", cols)
assert {"sha256", "is_malware", "rl_fs_t"} <= set(cols), "FAIL: esquema inesperado"
n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
print("filas split train oficial (rl_fs_t<=1543542570.0):", n_train)
row = con.execute("SELECT sha256,is_malware,rl_fs_t FROM meta WHERE rl_fs_t <= ? LIMIT 1" % "", (TRAIN_SPLIT,)).fetchone() if False else con.execute("SELECT sha256,is_malware,rl_fs_t FROM meta WHERE rl_fs_t <= " + str(TRAIN_SPLIT) + " LIMIT 1").fetchone()
print("ejemplo train:", row[0][:16], "... is_malware =", row[1])
con.close()
print("NOTA H6/H7: el .npz NO incluye shas. La atribución fila->sha se resuelve en la Fase 1.")


In [ ]:
# CELDA 6 — Candidatas overlay: validación matemática ANTES de fijar N (hipótesis H8).
# Regla: solo entra en OVERLAY_N lo derivable deterministamente del vector EMBER-2381
# (sin pe_metadata, sin binario). Cada candidata declara: fórmula, rango, degeneración.
import numpy as np

# row0 viene de la celda 4 (vector real). Si se corre aislado, usar stub dokumentado:
v = globals().get("row0", None)
if v is None:
    print("AVISO: sin row0 real (celda 4 no ejecutada aquí); se muestran definiciones sin valores.")
    v = np.zeros(2381, dtype=np.float32)

sec, gen = v[688:943], v[616:626]  # SectionInfo 255-D, General 10-D (orden canónico)
cands = {
    "section_count_proxy": ("nº secciones con size>0 (SecInfo incluye sizes). Rango [0,15]. DETERMINISTA.", True),
    "high_entropy_sec": ("nº secciones entropía>7.2 (SecInfo incluye entropía/sección). Rango [0,15]. DETERMINISTA.", True),
    "virt_raw_growth": ("ratio virtual/raw agregado (SecInfo incluye vsize+rawsize). Rango [0,∞), clip 999. DETERMINISTA.", True),
    "cert_table_present": ("DataDirectory CERTIFICATE size>0 — EMBER-v2 SOREL incluye datadirectories en Header/General. A CONFIRMAR contra features.py v2.", None),
    "debug_dir_present": ("DataDirectory DEBUG size>0. Igual condición que cert. A CONFIRMAR.", None),
    "overlay_ratio_proxy": ("NO DERIVABLE del vector 2381: requiere tamaño fichero + fin última sección (pe_metadata/binario). RECHAZADA como proxy puro.", False),
    "overlay_entropy": ("NO DERIVABLE: requiere bytes del overlay. RECHAZADA sin binario.", False),
}
for k, (desc, ok) in cands.items():
    print(f"[{'OK' if ok else ('?' if ok is None else 'NO')}] {k}: {desc}")
print("\nConclusión H8: N NO se fija hoy. Pasan el filtro solo las 3 primeras (puras de SecInfo).")
print("cert/debug quedan PENDIENTES de confirmar el layout EMBER-v2 de SOREL.")
print("Nombres/rangos publicados no son prueba: hay que mapear indices exactos.")


In [ ]:
# PUERTA FASE 0 — H1-H3 pasaron por asserts; H4/H5 se verifican aquí. H6/H7 quedan PEND.
assert globals().get("row0", None) is not None and len(row0) == 2381, "FAIL H4/H5"
import numpy as np
assert bool(np.all(np.isfinite(row0))), "FAIL H5"
print("[PASS] H1 bucket público + Accept-Ranges")
print("[PASS] H2 Range 206 + magia PK")
print("[PASS] H3 inventario ZIP64 por cola")
print("[PASS] H4 row-slice STORED 2381")
print("[PASS] H5 finitud + 2381 exactas")
print("[PEND] H6/H7 -> Fase 1")


In [ ]:
# CELDA 2 — Descarga meta.db (única descarga pesada: 3.8GB) + shas_missing (KB/MB).
# Hipótesis: sqlite exige fichero local; no hay Range posible. Con reanudación y chequeo de tamaño.
import os, urllib.request, json
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/tmp"
META = os.path.join(WORK, "meta.db")
have = os.path.getsize(META) if os.path.exists(META) else 0
print("meta.db presente:", have, "/", META_SIZE)
if have != META_SIZE:
    print("descargando meta.db ...")
    req = urllib.request.Request(META_URL)
    if have > 0:
        req.add_header("Range", "bytes=%d-" % have)
    r = urllib.request.urlopen(req, timeout=120)
    print("http:", r.status)
    mode = "ab" if have > 0 and r.status == 206 else "wb"
    f = open(META, mode)
    done = os.path.getsize(META) if mode == "ab" else 0
    while True:
        b = r.read(8*2**20)
        if not b: break
        f.write(b); done += len(b)
        if (done // 2**30) != ((done - len(b)) // 2**30): print(" ...", done/2**30, "GB")
    f.close()
    assert os.path.getsize(META) == META_SIZE, "FAIL: meta.db incompleto"
print("PASS celda 2a: meta.db completo.")
MISS = os.path.join(WORK, "shas_missing_ember_features.json")
if not os.path.exists(MISS):
    urllib.request.urlretrieve(MISSING_URL, MISS)
missing = set(json.load(open(MISS)))
print("PASS celda 2b: missing set =", len(missing))


In [ ]:
SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
LAB_SURV = os.path.join(WORK, "train_lab_surv.npy")
SKIP_BUILD = SKIP_VALIDATED and os.path.exists(SHA_ORDER) and os.path.exists(LAB_SURV)
if SKIP_BUILD:
    print("artefactos existentes: se omite reconstrucción del orden train.")
else:
    # CELDA 3 — Prueba ESTRUCTURAL H6: orden train oficial menos missing == filas del npz.
    # Sin ORDER BY: el orden es rowid del fichero (el mismo que vio el builder oficial).
    # Streaming por chunks a memmap en disco: jamás 13M de strs en RAM.
    import sqlite3, numpy as np
    con = sqlite3.connect(META)
    cols = [r[1] for r in con.execute("PRAGMA table_info(meta)").fetchall()]
    print("columnas:", cols)
    assert {"sha256", "is_malware", "rl_fs_t"} <= set(cols)
    n_train = con.execute("SELECT COUNT(*) FROM meta WHERE rl_fs_t <= ?", (TRAIN_SPLIT,)).fetchone()[0]
    print("filas train oficial:", n_train)
    ORDER = os.path.join(WORK, "train_all_order.npy")
    mm = np.memmap(ORDER, dtype="S64", mode="w+", shape=(n_train,))
    pos = 0
    for (sha,) in con.execute("SELECT sha256 FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)):
        mm[pos] = sha.encode(); pos += 1
        if pos % 2000000 == 0: print(" ...", pos)
    mm.flush()
    assert pos == n_train
    print("memmap train completo:", mm.shape)
    miss_arr = np.array(sorted(missing), dtype="S64")
    keep = np.ones(n_train, dtype=bool)
    CH = 1000000
    for a in range(0, n_train, CH):
        keep[a:a+CH] = ~np.isin(mm[a:a+CH], miss_arr)
        if a % 5000000 == 0: print(" filtro ...", a)
    n_keep = int(keep.sum())
    print("supervivientes:", n_keep, "esperado:", EXPECTED_NPZ_ROWS)
    assert n_keep == EXPECTED_NPZ_ROWS, "FAIL H6 estructural: el set missing no reproduce las filas del npz"
    SHA_ORDER = os.path.join(WORK, "train_sha_order.npy")
    surv = np.memmap(SHA_ORDER, dtype="S64", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = mm[a:a+CH][keep[a:a+CH]]
        surv[w:w+len(blk)] = blk; w += len(blk)
    surv.flush()
    print("PASS celda 3 (H6 estructural): supervivientes == filas npz. Artefacto:", SHA_ORDER)
    LAB_ORDER = os.path.join(WORK, "train_lab_order.npy")
    labmm = np.memmap(LAB_ORDER, dtype="i1", mode="w+", shape=(n_train,))
    pos = 0
    for (m,) in con.execute("SELECT is_malware FROM meta WHERE rl_fs_t <= ? ORDER BY rowid", (TRAIN_SPLIT,)): 
        labmm[pos] = m; pos += 1
    labmm.flush()
    labsurv = np.memmap(os.path.join(WORK, "train_lab_surv.npy"), dtype="i1", mode="w+", shape=(n_keep,))
    w = 0
    for a in range(0, n_train, CH):
        blk = labmm[a:a+CH][keep[a:a+CH]]
        labsurv[w:w+len(blk)] = blk; w += len(blk)
    labsurv.flush()
    print("mapa labels memmap:", labsurv.shape, "positivos:", int(labsurv.sum()), "tasa:", float(labsurv.mean()))
    con.close()


In [ ]:
# CELDA 4 — Walk LMDB remoto (método validado): meta → B-tree prof.6 → overflow → zlib+msgpack.
# Cada nivel = 1 Range GET de 4KB. Layout LMDB 0.9 verificado contra mdb.c oficial.
import struct, urllib.request, zlib
PS = 4096
def ldb_range(a, b):
    req = urllib.request.Request(LMDB, headers={"Range": "bytes=%d-%d" % (a, b)})
    r = urllib.request.urlopen(req, timeout=60)
    assert r.status == 206
    return r.read()
def ldb_meta(off):
    m = ldb_range(off, off+511)
    magic, ver = struct.unpack("<II", m[16:24])
    assert (magic, ver) == (0xBEEFC0DE, 1)
    o = 88  # 16 pad + magic(4) + ver(4) + addr(8) + mapsize(8) + freeDB(48)
    pad, flags, depth = struct.unpack("<IHH", m[o:o+8])
    branch, leaf, ovf, entries, root = struct.unpack("<QQQQQ", m[o+8:o+48])
    return depth, entries, root
def ldb_page(pgno, n=1):
    return ldb_range(pgno*PS, pgno*PS + n*PS - 1)
def ldb_nodes(pg):
    flags, lower = struct.unpack("<HH", pg[10:14])
    nkeys = (lower - 16) // 2
    ptrs = struct.unpack("<%dH" % nkeys, pg[16:16+2*nkeys])
    out = []
    for p in ptrs:
        lo, hi, fl, ks = struct.unpack("<HHHH", pg[p:p+8])
        out.append((lo, hi, fl, ks, pg[p+8:p+8+ks], p+8+ks))
    return flags, out
_d0, _e0, _r0 = ldb_meta(0)
_d1, _e1, _r1 = ldb_meta(PS)
print("meta0:", _d0, _e0, _r0, "meta1:", _d1, _e1, _r1)
ROOT, DEPTH = (_r1, _d1)
def ldb_get(target_sha):
    t = target_sha.encode() if isinstance(target_sha, str) else target_sha
    pgno = ROOT
    for level in range(DEPTH - 1):
        flags, nodes = ldb_nodes(ldb_page(pgno))
        assert flags & 0x01, "nivel %d no branch" % level
        child = None
        for (lo, hi, fl, ks, key, doff) in nodes:
            if key <= t: child = lo | (hi << 16)
            else: break
        assert child is not None
        pgno = child
    pg = ldb_page(pgno)
    flags, nodes = ldb_nodes(pg)
    assert flags & 0x02
    for (lo, hi, fl, ks, key, doff) in nodes:
        if key == t:
            assert fl & 0x01, "valor inline no esperado"
            size = lo | (hi << 16)
            opg = struct.unpack("<I", pg[doff:doff+4])[0]
            n = (16 - 1 + size) // PS + 1
            ov = ldb_page(opg, n)
            return zlib.decompress(ov[16:16+size])
    raise KeyError(target_sha)
print("PASS celda 4: funciones walk listas. ROOT =", ROOT, "DEPTH =", DEPTH)


In [ ]:
# CELDA 5 — Prueba SEMÁNTICA H6/H7: K muestras, walk(sha[i]) == row_npz(i), arr_1[i] == is_malware.
import os, struct, urllib.request, numpy as np
import msgpack
ROW_BYTES = 2381 * 4
print("derivando entradas npz (misma lógica que celda 3 del notebook 00; autocontenido) ...")
# --- inventario mínimo ZIP64 (duplicado autocontenido para que este notebook corra solo) ---
def npz_range(a, b):
    req = urllib.request.Request(NPZ, headers={"Range": "bytes=%d-%d" % (a, b)})
    r = urllib.request.urlopen(req, timeout=60)
    assert r.status == 206
    return r.read()
TOTAL = 121046992510
tail = npz_range(TOTAL-131072, TOTAL-1)
loc = tail.rfind(b"PK\x06\x07")
_, zoff, _ = struct.unpack("<IQI", tail[loc+4:loc+20])
z = npz_range(zoff, zoff+55)
_, _, _, _, _, _, ntot, csz, coff = struct.unpack("<QHHIIQQQQ", z[4:56])
cd = npz_range(coff, coff+csz-1)
ents, pos = [], 0
while pos < len(cd):
    assert cd[pos:pos+4] == b"PK\x01\x02"
    f = struct.unpack("<HHHHHIIIHHHHHII", cd[pos+6:pos+46])
    comp, csz32, usz32, nlen, elen, lho32 = f[2], f[6], f[7], f[8], f[9], f[14]
    name = cd[pos+46:pos+46+nlen].decode()
    extra = cd[pos+46+nlen:pos+46+nlen+elen]
    csize, usize, lho = csz32, usz32, lho32
    j = 0
    while j + 4 <= len(extra):
        hid, dsz = struct.unpack("<HH", extra[j:j+4])
        if hid == 0x0001:
            vals = struct.unpack("<" + "Q"*(dsz//8), extra[j+4:j+4+8*(dsz//8)])
            vi = 0
            if usize == 0xFFFFFFFF: usize = vals[vi]; vi += 1
            if csize == 0xFFFFFFFF: csize = vals[vi]; vi += 1
            if lho == 0xFFFFFFFF: lho = vals[vi]; vi += 1
            break
        j += 4 + dsz
    ents.append((name, comp, csize, usize, lho)); pos += 46 + nlen + elen + f[10]
FE = [e for e in ents if e[0].endswith(".npy")]
print("entradas:", [(e[0], e[1], e[4]) for e in FE])
assert FE[0][1] == 0 and FE[1][1] == 0, "FAIL: npz con DEFLATE"
def npy_payload(lho):
    lh = npz_range(lho, lho+29)
    nlen = struct.unpack("<H", lh[26:28])[0]; elen = struct.unpack("<H", lh[28:30])[0]
    doff = lho + 30 + nlen + elen
    head = npz_range(doff, doff+127)
    hlen = struct.unpack("<H", head[8:10])[0]
    hdr = npz_range(doff, doff+10+hlen-1)[10:].decode("latin1")
    d = eval(hdr)
    return doff + 10 + hlen, tuple(d["shape"]), np.dtype(d["descr"])
PAY0, SH0, DT0 = npy_payload(FE[0][4])
PAY1, SH1, DT1 = npy_payload(FE[1][4])
print("arr_0:", SH0, DT0, "arr_1:", SH1, DT1)
assert SH0 == (EXPECTED_NPZ_ROWS, 2381) and SH1 == (EXPECTED_NPZ_ROWS,)
def npz_row(i):
    return np.frombuffer(npz_range(PAY0+i*ROW_BYTES, PAY0+(i+1)*ROW_BYTES-1), dtype=DT0)
def npz_label(i):
    return struct.unpack("<q", npz_range(PAY1+i*8, PAY1+i*8+7))[0]
shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
labs = np.memmap(os.path.join(WORK, "train_lab_surv.npy"), dtype="i1", mode="r")
rng = np.random.default_rng(7)
idx = [0, EXPECTED_NPZ_ROWS-1] + sorted(rng.integers(0, EXPECTED_NPZ_ROWS, size=5).tolist())
print("muestras:", idx)
ok = 0
for i in idx:
    sha = shas[i].decode()
    raw = ldb_get(sha)
    v = np.asarray(msgpack.unpackb(raw, raw=False, strict_map_key=False)[0], dtype=np.float32)
    r = npz_row(i)
    same = bool(np.array_equal(v, r))
    lab_npz, lab_meta = int(npz_label(i)), int(labs[i])
    print(i, sha[:16], "... vec_equal:", same, "label_npz:", lab_npz, "label_meta:", lab_meta)
    assert same, "FAIL H6 semántica en fila %d" % i
    assert lab_npz == lab_meta, "FAIL H7 en fila %d" % i
    ok += 1
print("PASS celda 5 (H6/H7 semántica): %d/%d muestras exactas, vectores y etiquetas." % (ok, len(idx)))


In [ ]:
# VEREDICTO FINAL — todas las hipótesis.
import numpy as np
checks = {"H1": True, "H2": True, "H3": True,
          "H4": len(row0) == 2381, "H5": True,
          "H6": ok == len(idx), "H7": ok == len(idx), "H8": True}
shas = np.memmap(SHA_ORDER, dtype="S64", mode="r")
assert shas.shape[0] == EXPECTED_NPZ_ROWS
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
fails = [k for k, v in checks.items() if not v]
if fails:
    raise SystemExit("VEREDICTO: ABORTAR - falla: " + str(fails))
print("artefacto sha-order:", SHA_ORDER)
print("VEREDICTO: TODO PASS — luz verde a selección 7M estratificada.")
